# ML-04 — Search Intelligence Data Contract

## Topic

This notebook supports a content editor who must decide which pages to review for a possible refresh first. At the end of March 2026, it ranks pages by the risk that their organic-search clicks will materially decline in April. This is decision support: a future decline is a proxy for review priority, not proof that refreshing a page will cause recovery.

The notebook reads only the March and April 2026 warehouse partitions. A Hugging Face token is requested safely at runtime and is never stored in this notebook.

## 1. Unit of analysis + time window

**One row = one pseudonymized content page for one pseudonymized client, summarized over March 2026 at the March 31 decision point.** The raw warehouse fact is daily (`report_date × client_hash_id × content_hash_id`); this notebook aggregates the March daily records to the page-month decision grain.

**Feature window:** March 1–31, 2026. **Decision moment:** after March 31. **Outcome window:** April 1–30, 2026. The label is `1` when April GSC clicks are less than 80% of March GSC clicks, for pages with at least 20 March clicks. The threshold reduces small-count noise; it is a transparent policy choice, not ground truth.

In [1]:
import os
from getpass import getpass
from huggingface_hub import get_token
from pathlib import Path

import duckdb
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Set HF_TOKEN in your environment or enter it only in this hidden prompt. Never put a token in a notebook cell.
HF_TOKEN = os.environ.get('HF_TOKEN') or get_token() or getpass('Hugging Face READ token (hf_...): ')
if not HF_TOKEN.startswith('hf_'):
    raise ValueError('A Hugging Face READ token is required; do not paste it into the notebook.')

repo_root = Path.cwd()
if not (repo_root / 'skills').is_dir():
    repo_root = repo_root.parent.parent
extension_dir = repo_root / 'work' / 'outputs' / '.duckdb_extensions'
extension_dir.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(config={'extension_directory': str(extension_dir)})
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
print('Connected. Queries below read only the March and April 2026 partitions.')

C:\Users\makaah\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected. Queries below read only the March and April 2026 partitions.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields used | Why |
|---|---|---|
| Features | March `gsc_clicks`, `gsc_impressions`, derived March CTR, March `gsc_avg_position`, and March days with impressions | All are measured during the completed March feature window, before the April outcome. |
| Label / proxy | `is_april_click_decline` = April clicks < 80% of March clicks, where March clicks ≥ 20 | It is an observed future click-decline proxy for which pages deserve editorial review first. It is not evidence that a refresh would cause improvement. |
| Context | `client_hash_id`, `content_hash_id`, `report_date` | These pseudonymous keys define, group, and audit the data; they are not model inputs. |
| Excluded | April clicks and all other April metrics | They happen after the March 31 decision point and would leak the answer. |
| Excluded | `ga4_data_available = FALSE` GA4 values | Those GA4 values are zero-filled before a client's analytics coverage begins, so zero does not mean zero engagement. |

## 3. Verify it with queries

The next three cells are the **three required verification queries**. They check the post-aggregation grain, the raw March slice size and dates, and GA4 availability using `IS TRUE`.

In [2]:
# Verification query 1 — grain: an empty result means each client-page has one March row.
grain_check = con.sql(f"""
    WITH march_pages AS (
        SELECT client_hash_id, content_hash_id
        FROM {MARCH}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, COUNT(*) AS rows_per_page_month
    FROM march_pages
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
grain_check

,client_hash_id,content_hash_id,rows_per_page_month


In [3]:
# Verification query 2 — the size and observed date span of the March daily slice.
slice_check = con.sql(f"""
    SELECT COUNT(*) AS march_daily_rows,
           MIN(report_date) AS first_report_date,
           MAX(report_date) AS last_report_date
    FROM {MARCH}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
slice_check

,march_daily_rows,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


In [4]:
# Verification query 3 — rows with real GA4 coverage. IS TRUE avoids treating zero-filled rows as observations.
availability_check = con.sql(f"""
    SELECT COUNT(*) AS march_rows_with_ga4_available
    FROM {MARCH}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
      AND ga4_data_available IS TRUE
""").df()
availability_check

,march_rows_with_ga4_available
0,413966


### Five-feature frame

1. **`march_clicks`** — knowable at the decision moment because the complete March click total is observed by March 31.
2. **`march_impressions`** — knowable at the decision moment because it is measured only during March.
3. **`march_ctr`** — knowable at the decision moment because it is calculated only from March clicks and March impressions.
4. **`march_avg_position`** — knowable at the decision moment because it summarizes March search positions before April occurs.
5. **`march_days_with_impressions`** — knowable at the decision moment because it counts March days on which the page had observed search impressions.

The feature query below uses April only to construct the future label. No April-derived field is retained as an honest feature.

In [5]:
features = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id,
               SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
               SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
               AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
               SUM(CASE WHEN COALESCE(gsc_impressions, 0) > 0 THEN 1 ELSE 0 END) AS march_days_with_impressions
        FROM {MARCH}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
        GROUP BY 1, 2
    ),
    april AS (
        SELECT client_hash_id, content_hash_id,
               SUM(COALESCE(gsc_clicks, 0)) AS april_clicks
        FROM {APRIL}
        WHERE report_date >= DATE '2026-04-01' AND report_date < DATE '2026-05-01'
        GROUP BY 1, 2
    )
    SELECT m.client_hash_id, m.content_hash_id,
           m.march_clicks,
           m.march_impressions,
           m.march_clicks / NULLIF(m.march_impressions, 0)::DOUBLE AS march_ctr,
           m.march_avg_position,
           m.march_days_with_impressions,
           CASE WHEN COALESCE(a.april_clicks, 0) < 0.8 * m.march_clicks THEN 1 ELSE 0 END AS is_april_click_decline
    FROM march m
    LEFT JOIN april a USING (client_hash_id, content_hash_id)
    WHERE m.march_clicks >= 20
""").df()

feature_cols = ['march_clicks', 'march_impressions', 'march_ctr', 'march_avg_position', 'march_days_with_impressions']
model_data = features.dropna(subset=feature_cols).copy()
print(f'Feature frame: {len(model_data):,} eligible client-page rows')
print(f'April-decline proxy rate: {model_data.is_april_click_decline.mean():.1%}')
model_data[feature_cols + ['is_april_click_decline']].head()

Feature frame: 10,100 eligible client-page rows
April-decline proxy rate: 46.6%


,march_clicks,march_impressions,march_ctr,march_avg_position,march_days_with_impressions,is_april_click_decline
0,37.0,6051.0,0.006115,5.415020,22.0,0
1,40.0,1764.0,0.022676,9.743764,31.0,1
2,32.0,5350.0,0.005981,11.147605,31.0,0
3,202.0,34689.0,0.005823,3.817858,31.0,0
4,28.0,11030.0,0.002539,6.514271,31.0,1


### Deliberate leakage experiment

I now add **one label-derived column**, `leak_target_copy`, which is an exact copy of the April-decline label. This is intentionally invalid: the label is only known after April ends. The quick score should become perfect or nearly perfect because the model has been handed the answer. I then delete that column and retain only the honest score.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    model_data[feature_cols], model_data['is_april_click_decline'],
    test_size=0.25, random_state=42, stratify=model_data['is_april_click_decline']
)

# Intentionally wrong: this feature is derived from the future label itself.
X_train_leaky = X_train.assign(leak_target_copy=y_train.to_numpy())
X_test_leaky = X_test.assign(leak_target_copy=y_test.to_numpy())
leaky_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train_leaky, y_train)
leaky_accuracy = accuracy_score(y_test, leaky_model.predict(X_test_leaky))
print(f'INTENTIONALLY LEAKY accuracy: {leaky_accuracy:.3f}')

# Delete the leak and keep this honest comparison.
del X_train_leaky, X_test_leaky
honest_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
honest_accuracy = accuracy_score(y_test, honest_model.predict(X_test))
print(f'Honest accuracy after deleting the leak: {honest_accuracy:.3f}')
print('The honest model uses only the five March features listed above.')

INTENTIONALLY LEAKY accuracy: 1.000
Honest accuracy after deleting the leak: 0.596
The honest model uses only the five March features listed above.


## 4. Data limits

This slice cannot show that refreshing a page causes recovery: it observes click movement, not editorial interventions or causal effects. Client history is also unbalanced, and GA4 is unavailable for some client-date rows; therefore the queue is a measured, directional decision-support signal rather than a universal or causal recommendation.

## 5. Self-check

- [x] I stated one clear client-page March decision grain and separate March feature / April outcome windows.
- [x] I classified every field used as a feature, label/proxy, context, or exclusion.
- [x] I included exactly three verification queries and used `ga4_data_available IS TRUE` for availability.
- [x] I built exactly five March-only features and explained when each is knowable.
- [x] I demonstrated a label-derived leak, deleted it, and retained the honest result.
- [ ] I executed this notebook top to bottom with a Hugging Face READ token, checked the visible outputs, and committed it.